In [2]:
import pandas as pd

path = 'raw_data/all_confirmed.csv'
df = pd.read_csv(path)

df = df.rename(columns={'Automated Persona Match': 'Auto IP'})

In [3]:
df.columns

Index(['Subject', 'Full Name', 'Title', 'Job Level', 'Meeting Plan',
       'Meeting Recap', 'Notes', 'Ideal Persona', 'Auto IP', 'Account Type',
       'ICP 2.0', 'Target Account', 'Account Name', 'Account Owner',
       'Attending Sales Rep', 'Activity ID', 'Created Date', 'Email',
       'Contact ID', 'Account ID', 'Account Region Level 1',
       'Last Modified By', 'Last Modified Date', 'Owner Region Level 2',
       'Assigned', 'Date', 'Territory Region Level 2'],
      dtype='object')

In [ ]:



# ---------------------------------------------------------------
# Helper: turn 1/0, True/False, "true"/"false" (any case) into clean booleans.
# Blanks and anything unexpected become False.
# ---------------------------------------------------------------
def to_bool(series):
    numeric = pd.to_numeric(series, errors='coerce')          # 1, 1.0, "1", True -> 1 ; text -> NaN
    text = series.astype(str).str.strip().str.lower()         # "True " -> "true"
    return (numeric == 1) | (text == 'true')                  # True if either test says yes


# ---------------------------------------------------------------
# CHECK 1 - Account Check: is this the right kind of account?
# ---------------------------------------------------------------

# ICP prospects: Former/Potential Customer AND flagged as a target account
cond1 = (
    df['Account Type'].isin(['Former Customer', 'Potential Customer'])
    & to_bool(df['Target Account'])
)

# Customers: only allowed if the activity is an Orlando Experience registration
cond2 = (
    df['Subject'].str.contains('Registered for Orlando Experience', case=False, na=False)
    & df['Account Type'].isin(['Customer', 'Indirect Customer', 'MSP Customer'])
)

# Pass if either condition is met
df['Account Check'] = cond1 | cond2


# ---------------------------------------------------------------
# CHECK 2 - Experience Check: are experience registrations documented?
# ---------------------------------------------------------------
is_experience = df['Subject'].str.contains('Registered for Orlando Experience', case=False, na=False)
has_sf_link = df['Meeting Recap'].str.contains('nexthink.lightning.force', case=False, na=False, regex=False)

# Non-experience rows pass automatically; experience rows need the link
df['Experience Check'] = ~is_experience | has_sf_link


# ---------------------------------------------------------------
# CHECK 3 - Persona Check: at least 1 persona match per activity
# ---------------------------------------------------------------

ids_qualify_ip = [
    '003NN00000XIo51', '003NN00000OmnDX', '003NN00000P6guF', '003NN00000XMHGy',
    '003NN00000Y28U9', '003NN00000XDEF5', '003NN00000XDr6m', '003NN00000YDSxU',
    '003NN00000Y7WcJ', '003NN00000Pd5RF', '003NN00000YGh2Q', '003NN00000Y2EzUYAV',
    '003NN00000YOXbz', '0035700001yk3mn', '003NN00000ZfBGJ', '003c40000052onR', '003NN00000Z3H52',
    '003NN00000Z3H52YAF', '003NN00000XtnMt', '003NN00000YaFub', '0032p000030SHt8', '003NN00000XsfCt',
    '0037U00000G4eAU', '003NN00000WkXxp', '003NN00000WkY5v','003NN00000Xhaej', '003NN00000Wm1Nc',
    '003NN00000UcSMz', '003NN00000XMtEF','003NN00000XZGzp','003c400000ObEpSAAV', '0036900003oKxwR',
    '003NN00000Wihrm','003NN00000YOnLo', '003NN00000Xtug8','003NN00000WKYXl',
    '003NN00000WFUUp', '003NN00000Yy37o', '003NN00000Kg7GO','003NN00000TSPQz',
    '003c4000006JttO', '003NN00000YZGXl', '003NN00000WgOhv','0035700001xRXPb',
    '003NN00000YIszH', '003NN00000Z7KCU', '003NN00000Xx8pl','003NN00000XJpwg'






]

# Convert 1/0 to True/False
df['Ideal Persona'] = to_bool(df['Ideal Persona'])
df['Auto IP'] = to_bool(df['Auto IP'])

# Start from Auto IP, then force True for the listed contacts
df['Auto IP adjusted'] = df['Auto IP'] | df['Contact ID'].isin(ids_qualify_ip)

# Any persona match within the activity?
df['Persona Check'] = df.groupby('Activity ID')['Auto IP adjusted'].transform('any')

# ---------------------------------------------------------------
# FINAL - must pass all three checks
# ---------------------------------------------------------------
df['Qualified'] = df['Account Check'] & df['Experience Check'] & df['Persona Check']


# Optional: one row per activity, True if any of its rows qualified
activity_summary = (
    df.groupby('Activity ID')['Qualified']
      .any()
      .reset_index()
)

In [5]:
check_df = df[['Activity ID', 'Contact ID', 'Assigned', 'Full Name', 'Title', 'Ideal Persona', \
               'Auto IP', 'Auto IP adjusted','Account Name', 'Account Type', \
                'Target Account', 'Meeting Plan','Meeting Recap', 'Notes', \
                'Account Check', 'Experience Check', 'Persona Check', 'Qualified']]

In [8]:
check_df.to_csv('processed_data/encounter_checks_output.csv', index=False, encoding='utf-8-sig')